In [13]:
import time
import xgboost as xgb
import numpy as np
import joblib
from pathlib import Path
from credit_risk.dataset import load_splits, AFTER_EDA
from credit_risk.features import prep_one_split, NUMERICAL_COLS

In [2]:
cwd = Path.cwd()
project_root = cwd.parent
xgb_path = project_root / 'models' / 'tuned_xgb'

### Latency

In [3]:
_, _, test_df, _ = load_splits(path=AFTER_EDA)

2026-07-05 16:09:01.871 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-07-05 16:09:01.883 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-07-05 16:09:02.190 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [4]:
# loading the model
xgb_model = joblib.load(xgb_path / 'model.pkl')

In [16]:
pt_x_test, y_test = prep_one_split(test_df)

2026-07-05 20:00:05.466 | INFO     | credit_risk.features:prep_one_split:217 - Inside Function: prep_one_split
2026-07-05 20:00:05.467 | INFO     | credit_risk.features:sorting_with_issue_d:143 - Sorting the dataframe wrt to issue_d...
2026-07-05 20:00:05.699 | INFO     | credit_risk.features:sorting_with_issue_d:148 - Sorted successfully!
2026-07-05 20:00:05.700 | INFO     | credit_risk.features:split_target_and_features:153 - Inside Function: split_target_and_features
2026-07-05 20:00:05.700 | INFO     | credit_risk.features:split_target_and_features:154 - Splitting the target and the features...
2026-07-05 20:00:05.700 | INFO     | credit_risk.features:split_target_and_features:157 - features and target are splitted successfully...
2026-07-05 20:00:05.700 | INFO     | credit_risk.features:add_credit_yrs:170 - Inside Function: add_credit_yrs
2026-07-05 20:00:05.700 | INFO     | credit_risk.features:add_credit_yrs:172 - Adding the feature 'credit_yrs'
2026-07-05 20:00:05.709 | INFO   

In [6]:
pt_x_test[:1].shape

(1, 65)

In [7]:
# latency check
# xgb

# warm up
for _ in range(10):
    X_test = xgb_model[0].transform(pt_x_test[:1])
    _ = xgb_model[1].predict_proba(X_test)
    
n_trials = 100
times = []
for _ in range(n_trials):
    start = time.perf_counter()
    X_test = xgb_model[0].transform(pt_x_test[:1])
    _ = xgb_model[1].predict_proba(X_test)
    end = time.perf_counter()
    times.append((end - start) * 1000)
    
print(f"median: {np.median(times):.3f} ms")
print(f'P95: {np.percentile(times, 95):.3f} ms')

median: 1.430 ms
P95: 1.771 ms


In [8]:
# batch latency
# xgb

# warm up
for _ in range(3):
    X_test = xgb_model[0].transform(pt_x_test[:1000])
    _ = xgb_model[1].predict_proba(X_test)
    
n_trials = 50
times = []
for _ in range(n_trials):
    start = time.perf_counter()
    X_test = xgb_model[0].transform(pt_x_test[:1000])
    _ = xgb_model[1].predict_proba(X_test)
    end = time.perf_counter()
    
    times.append((end - start) * 1000)
    
print(f"Median: {np.median(times):.3f} ms")
print(f"per-row: {(np.median(times)/1000):.3f} ms")

Median: 4.197 ms
per-row: 0.004 ms


### Size

In [12]:
xgb_size = (xgb_path / 'model.pkl').stat().st_size / (1024**2)
print(f"XGB (preprocessor + model): {xgb_size:.2f} MB")

XGB (preprocessor + model): 5.40 MB


### Perturbation Test

In [30]:
np.random.seed(42)
perturb_test_set = pt_x_test.copy()

cols_to_purturb = [col for col in NUMERICAL_COLS if col in perturb_test_set.columns]

noise = np.random.normal(1, 0.1, size=(len(perturb_test_set), len(cols_to_purturb)))
perturb_test_set[cols_to_purturb] = perturb_test_set[cols_to_purturb].values * noise

In [31]:
perturb_test_set = xgb_model[0].transform(perturb_test_set)
xgb_perturb_pp = xgb_model[1].predict_proba(perturb_test_set)[:,1]

In [32]:
from sklearn.metrics import average_precision_score

xgb_perturbed_pr = average_precision_score(y_true=y_test, y_score=xgb_perturb_pp)
xgb_clean_pr = 0.3105 
print(f"XGBoost clean:     {xgb_clean_pr:.4f}")
print(f"XGBoost perturbed: {xgb_perturbed_pr:.4f}")
print(f"Delta: {xgb_perturbed_pr - xgb_clean_pr:+.4f}")

XGBoost clean:     0.3105
XGBoost perturbed: 0.2925
Delta: -0.0180


### Directional Expectation Test

In [48]:
# DTI
change = 20
de_test_set = pt_x_test.sample(n=500, random_state=42)
non_de_test_set = de_test_set.copy()
de_test_set['dti'] += change

In [49]:
de_test_set = xgb_model[0].transform(de_test_set)
xgb_de_pp = xgb_model[1].predict_proba(de_test_set)[:,1]

non_de_test_set = xgb_model[0].transform(non_de_test_set)
xgb_non_de_pp = xgb_model[1].predict_proba(non_de_test_set)[:,1]

In [50]:
avg_delta_pp = (xgb_de_pp - xgb_non_de_pp).mean()

print(f"For XGB: added {change} in dti: Avg delta in predicted probs(changed_dti - unchanged_dti): {avg_delta_pp:.4f}")

For XGB: added 20 in dti: Avg delta in predicted probs(changed_dti - unchanged_dti): 0.0384


In [57]:
# annual_inc
change = 10000
de_test_set = pt_x_test.sample(n=500, random_state=42)
non_de_test_set = de_test_set.copy()
de_test_set['annual_inc'] += change

In [58]:
de_test_set = xgb_model[0].transform(de_test_set)
xgb_de_pp = xgb_model[1].predict_proba(de_test_set)[:,1]

non_de_test_set = xgb_model[0].transform(non_de_test_set)
xgb_non_de_pp = xgb_model[1].predict_proba(non_de_test_set)[:,1]

In [59]:
avg_delta_pp = (xgb_de_pp - xgb_non_de_pp).mean()

print(f"For XGB: added {change} in annual inc: Avg delta in predicted probs(changed_dti - unchanged_dti): {avg_delta_pp:.4f}")

For XGB: added 10000 in annual inc: Avg delta in predicted probs(changed_dti - unchanged_dti): -0.0092
